# Topic 33: a tiny decoder-only language model

Everything from the Topic 33 slides, in one short notebook, trained on a little real English:

* a **word-level tokenizer** (the textbook version: a fixed dictionary, `<unk>` for rare words),
* `<sos>` / `<eos>` / `<pad>`, and the **target = the input shifted left by one**,
* the **next-token loss**, with `<pad>` targets left out,
* a **decoder-only** model: Transformer blocks with a **causal mask** and no encoder,
* the **generation loop** with greedy, temperature, top-k and top-p.

**Runtime:** about 3 minutes in all. Training stops after `TRAIN_SECONDS` (150 s) on any machine; a GPU
(*Runtime > Change runtime type > T4 GPU*) just fits more steps into that time.

**Data:** the first 5,000 stories of [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories)
(Eldan and Li, 2023; CDLA-Sharing-1.0): short stories in simple English. One 19 MB file is downloaded.

In [ ]:
import collections, json, os, random, re, time, urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); random.seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_STORIES = 5000          # how much text to use
VOCAB_SIZE = 2000         # dictionary: the 2,000 most common words, everything else is <unk>
TRAIN_SECONDS = 150       # training stops here, whatever the hardware

## 1. Data: a few thousand short stories, cut into sentences

In [ ]:
# 1. Data: a few thousand short stories, cut into sentences ------------------------------------
URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt"
if not os.path.exists("TinyStories-valid.txt"):
    urllib.request.urlretrieve(URL, "TinyStories-valid.txt")
stories = open("TinyStories-valid.txt", encoding="utf-8").read().split("<|endoftext|>")[:N_STORIES]


def words_of(text):                     # "Spot saw the car." -> ['spot', 'saw', 'the', 'car', '.']
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[.,!?\"]", text.lower())


sentences = []
for story in stories:
    for s in re.split(r"(?<=[.!?])\s+", story.strip()):
        w = words_of(s)
        if 3 <= len(w) <= 30:
            sentences.append(w)
random.shuffle(sentences)
print(len(stories), "stories ->", len(sentences), "sentences, e.g.", " ".join(sentences[0]))

## 2. Tokenizer: one word = one token
A fixed dictionary of the 2,000 most common words; every other word becomes `<unk>`. (Real GPT models use subword pieces instead, slide 10.)

In [ ]:
# 2. Tokenizer: one word = one token, a fixed dictionary, <unk> for the rest -------------------
PAD, SOS, EOS, UNK = 0, 1, 2, 3
counts = collections.Counter(w for s in sentences for w in s)
itos = ["<pad>", "<sos>", "<eos>", "<unk>"] + [w for w, _ in counts.most_common(VOCAB_SIZE - 4)]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
n_unk = sum(c for w, c in counts.items() if w not in stoi)
print(f"{len(counts)} different words; dictionary V = {V}; "
      f"{100 * n_unk / sum(counts.values()):.1f}% of the words become <unk>")


def encode(words):                      # ['the', 'cat', ...] -> [<sos>, the, cat, ..., <eos>]
    return [SOS] + [stoi.get(w, UNK) for w in words] + [EOS]


data = [torch.tensor(encode(s)) for s in sentences]
val, train = data[:1000], data[1000:]


def batch(seqs):
    seq = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=PAD)   # (B, T+1)
    return seq[:, :-1].to(dev), seq[:, 1:].to(dev)   # input, and target = input shifted left by one

## 3. Model: decoder-only
`nn.TransformerEncoderLayer` is self-attention + feed-forward. With a **causal mask** and no cross-attention, a stack of them is exactly a decoder-only model. Note the shapes: `(B, T)` ids in, `(B, T, V)` logits out.

In [ ]:
# 3. Model: decoder-only = Transformer blocks with a causal mask, and no encoder ----------------
class TinyLM(nn.Module):
    def __init__(self, V, d=128, heads=4, layers=3, max_len=32):
        super().__init__()
        self.tok = nn.Embedding(V, d)            # token embedding: V x d
        self.pos = nn.Embedding(max_len, d)      # learned position embedding
        block = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d, dropout=0.1,
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(d)
        self.out = nn.Linear(d, V)               # one score (logit) per dictionary entry

    def forward(self, ids):                      # ids: (B, T)
        T = ids.size(1)
        x = self.tok(ids) + self.pos(torch.arange(T, device=ids.device))      # (B, T, d)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=ids.device)
        x = self.blocks(x, mask=causal, is_causal=True)   # position t sees positions <= t only
        return self.out(self.ln(x))                       # (B, T, V) logits


model = TinyLM(V).to(dev)
print("parameters:", sum(p.numel() for p in model.parameters()))
x, y = batch(train[:4])
print("input ids:", tuple(x.shape), " logits:", tuple(model(x).shape))   # (B, T) -> (B, T, V)
# <pad> only ever comes after <eos>, and the causal mask stops a real token from looking ahead,
# so no real token ever attends to a <pad>: the causal mask is enough here.


def loss_of(x, y):
    logits = model(x)                                             # (B, T, V)
    return F.cross_entropy(logits.reshape(-1, V), y.reshape(-1),  # B*T classification problems
                           ignore_index=PAD)                      # a <pad> target is not graded


@torch.no_grad()
def val_loss():
    model.eval()
    l = sum(loss_of(*batch(val[i:i + 250])).item() for i in range(0, len(val), 250)) / (len(val) // 250)
    model.train()
    return l

## 4. Training: predict the next token at every position
The loss is `B*T` classification problems at once, with `<pad>` targets left out (`ignore_index`). Uniform guessing over `V` words would give `ln V`.

In [ ]:
# 4. Training: next-token cross-entropy, for TRAIN_SECONDS ------------------------------------
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
log = {"V": V, "n_sentences": len(sentences), "curve": []}
print(f"step     0  val loss {val_loss():.3f}   (uniform guessing: ln V = {torch.log(torch.tensor(float(V))):.3f})")
t0, step, next_report = time.time(), 0, 15
while time.time() - t0 < TRAIN_SECONDS:
    x, y = batch(random.sample(train, 64))
    loss = loss_of(x, y)
    opt.zero_grad(); loss.backward(); opt.step()
    step += 1
    if time.time() - t0 >= next_report:
        v = val_loss()
        log["curve"].append((step, round(time.time() - t0), loss.item(), v))
        print(f"step {step:5d}  train loss {loss.item():.3f}  val loss {v:.3f}  ({time.time() - t0:.0f} s)")
        next_report += 15
log["steps"], log["seconds"] = step, time.time() - t0
print(f"trained {step} steps in {log['seconds']:.0f} s on {dev}")

## 5. Generation: the loop from the slides
Only the **last** position's logits are used; the chosen token is appended and fed back in. Try other prompts and settings.

In [ ]:
# 5. Generation: the loop of the slides, with greedy / temperature / top-k / top-p --------------
def choose(logits, greedy=False, temperature=1.0, top_k=None, top_p=None):
    if greedy:
        return int(logits.argmax())
    logits = logits / temperature
    if top_k:                                                     # keep the k most likely
        kth = logits.topk(top_k).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p:                                                     # keep the smallest set adding up to p
        p, order = logits.softmax(-1).sort(descending=True)
        drop = (p.cumsum(0) - p) >= top_p                         # the set was already full before it
        logits[order[drop]] = float("-inf")
    return int(torch.multinomial(logits.softmax(-1), 1))          # -inf -> probability 0, renormalized


@torch.no_grad()
def generate(prompt, max_new_tokens=25, **how):
    ids = [SOS] + [stoi.get(w, UNK) for w in words_of(prompt)]
    for _ in range(max_new_tokens):
        logits = model(torch.tensor([ids[-32:]], device=dev))[0, -1]   # last position only
        nxt = choose(logits, **how)
        ids.append(nxt)
        if nxt == EOS:
            break
    return " ".join(itos[i] for i in ids[1:] if i != EOS)


model.eval()
log["samples"] = {}
for prompt in ["once upon a time", "the little dog", "she was very"]:
    log["samples"][prompt] = {"greedy": generate(prompt, greedy=True)}
    print(f"greedy       | {log['samples'][prompt]['greedy']}")
torch.manual_seed(1)
for name, how in [("T=1", dict(temperature=1.0)), ("top-k=10", dict(top_k=10)),
                  ("top-p=0.9", dict(top_p=0.9)), ("T=2", dict(temperature=2.0))]:
    outs = [generate("once upon a time", **how) for _ in range(2)]
    log["samples"][name] = outs
    for o in outs:
        print(f"{name:12s} | {o}")

## 6. Testing a model that only continues text
Three quick checks: the most likely next tokens, a real sentence against the same words shuffled, and perplexity = `exp(val loss)`.

In [ ]:
# 6. Testing a model that only continues text --------------------------------------------------
@torch.no_grad()
def next_tokens(prompt, k=5):                 # the top-k next tokens and their probabilities
    ids = [SOS] + [stoi.get(w, UNK) for w in words_of(prompt)]
    p = model(torch.tensor([ids], device=dev))[0, -1].softmax(-1)
    top = p.topk(k)
    return [(itos[i], round(v, 3)) for v, i in zip(top.values.tolist(), top.indices.tolist())]


@torch.no_grad()
def sentence_loss(text):                      # average -log p over the sentence's tokens
    x, y = batch([torch.tensor(encode(words_of(text)))])
    return loss_of(x, y).item()


log["next"] = {}
for prompt in ["once upon a", "the cat sat on the", "she was very", "they played in the"]:
    log["next"][prompt] = next_tokens(prompt)
    print(f"{prompt!r:22} -> {log['next'][prompt]}")

log["pairs"] = []
for good in ["the dog ran to the park .", "she smiled and said thank you .", "he was sad because he lost his toy ."]:
    w = good.split()
    bad = " ".join(random.Random(0).sample(w, len(w)))
    log["pairs"].append((good, sentence_loss(good), bad, sentence_loss(bad)))
    print(f"loss {sentence_loss(good):.2f}  {good!r}\nloss {sentence_loss(bad):.2f}  {bad!r}  (shuffled)")

log["perplexity"] = float(torch.exp(torch.tensor(val_loss())))
print(f"perplexity = exp(val loss) = {log['perplexity']:.1f}: as unsure as picking among that many words, "
      f"not among all {V}")

## 7. It is not a chat model
A question is just more text to continue. Turning this into a chat model is the next part of the lecture.

In [ ]:
# Not a chat model: a question is just more text to continue.
log["question"] = generate("what is your name ?", greedy=True)
print("what is your name ? ->", log["question"])